# Spherical Delaunay README hero

This slow, manually refreshed artifact builds a deterministic spherical Delaunay triangulation of $S^2 \subset \mathbb{R}^3$ with the Rust prototype and renders its actual simplices as a transparent README hero. It is intentionally excluded from routine notebook checks.

In [ ]:
"""Generate and parse the deterministic Rust S2 triangulation artifact."""

import json
import os
import subprocess
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d.art3d import Poly3DCollection


def find_repo_root(start: Path) -> Path:
    """Find the repository root from a notebook or nbconvert working directory."""
    for candidate in (start, *start.parents):
        if (candidate / "Cargo.toml").is_file() and (candidate / "justfile").is_file():
            return candidate
    message = "run this notebook from inside the delaunay repository"
    raise RuntimeError(message)


def tracked_figure_path_from_env(name: str, root: Path, default: Path, expected_relative: Path) -> Path:
    """Use scratch output unless an exact tracked figure path is enabled."""
    configured = os.environ.get(name)
    if configured is None:
        return default
    if not configured.strip():
        raise ValueError(f"{name} must not be empty")
    path = Path(configured)
    expected = (root / expected_relative).resolve()
    if path.is_absolute() or (root / path).resolve() != expected:
        raise ValueError(f"{name} must be the repo-relative path {expected_relative.as_posix()!r}")
    return expected


ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = ROOT / "target" / "notebooks" / "02_spherical_hero"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_PATH = OUTPUT_DIR / "spherical_hero.json"
FIGURE_PATH = tracked_figure_path_from_env(
    "DELAUNAY_SPHERICAL_HERO_FIGURE",
    ROOT,
    OUTPUT_DIR / "delaunay_spherical_readme.png",
    Path("docs/assets/readme/delaunay_spherical_readme.png"),
)
VERTEX_COUNT = int(os.environ.get("DELAUNAY_SPHERICAL_HERO_VERTICES", "160"))
if not 4 <= VERTEX_COUNT <= 2**32 - 1:
    raise ValueError(f"spherical hero vertex count must be in [4, 2^32 - 1], got {VERTEX_COUNT}")

binary = ROOT / "target" / "perf" / "delaunay"
if not binary.is_file():
    raise FileNotFoundError(f"build the notebook CLI first: {binary}")
command = [
    str(binary),
    "spherical-hero",
    "--vertices",
    str(VERTEX_COUNT),
    "--output",
    str(JSON_PATH),
]
result = subprocess.run(  # noqa: S603
    command,
    cwd=ROOT,
    check=False,
    capture_output=True,
    text=True,
    timeout=1800,
)
if result.returncode != 0:
    raise RuntimeError(f"spherical hero generation failed ({result.returncode}): {' '.join(command)}\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}")
with JSON_PATH.open(encoding="utf-8") as handle:
    artifact: Any = json.load(handle)
if not isinstance(artifact, dict):
    raise TypeError(f"spherical hero artifact must be a JSON object: {JSON_PATH}")
expected_metadata = {
    "schema": "delaunay.spherical_hero",
    "schema_version": 1,
    "intrinsic_dimension": 2,
    "ambient_dimension": 3,
}
for field, expected in expected_metadata.items():
    if artifact.get(field) != expected:
        raise ValueError(f"unexpected spherical hero {field}: expected {expected!r}, got {artifact.get(field)!r}")
vertices = np.asarray(artifact.get("vertices"), dtype=np.float64)
simplices = np.asarray(artifact.get("simplices"), dtype=np.int64)
if vertices.shape != (VERTEX_COUNT, 3):
    raise ValueError(f"expected {(VERTEX_COUNT, 3)} vertices, got {vertices.shape}")
if not np.isfinite(vertices).all():
    message = "spherical hero vertices must contain only finite coordinates"
    raise ValueError(message)
squared_norms = np.einsum("ij,ij->i", vertices, vertices)
if not np.allclose(squared_norms, 1.0, rtol=0.0, atol=1.0e-12):
    message = "spherical hero vertices must lie on the unit sphere"
    raise ValueError(message)
if simplices.ndim != 2 or simplices.shape[1] != 3:
    raise ValueError(f"expected triangular simplices, got {simplices.shape}")
if len(simplices) != 2 * VERTEX_COUNT - 4:
    raise ValueError(f"expected {2 * VERTEX_COUNT - 4} closed-sphere triangles, got {len(simplices)}")
if np.any(simplices < 0) or np.any(simplices >= VERTEX_COUNT):
    message = "spherical hero simplices reference out-of-range vertex indices"
    raise ValueError(message)
if np.any(np.diff(np.sort(simplices, axis=1), axis=1) == 0):
    message = "spherical hero simplices must contain three distinct vertices"
    raise ValueError(message)
print(f"S2 triangulation: {len(vertices)} vertices, {len(simplices)} triangles")

In [ ]:
"""Render the actual spherical simplices as the transparent README hero."""

triangles = vertices[simplices]
centroids = triangles.mean(axis=1)
light_direction = np.asarray([0.35, -0.45, 0.82], dtype=np.float64)
light_direction /= np.linalg.norm(light_direction)
intensity = np.clip(0.28 + 0.72 * (centroids @ light_direction), 0.12, 1.0)
base_color = np.asarray([0.10, 0.55, 0.88], dtype=np.float64)
facecolors = np.column_stack(
    (
        base_color[None, :] * intensity[:, None],
        np.full(len(triangles), 0.88),
    )
)

figure = plt.figure(figsize=(12, 6), facecolor="none")
axis = figure.add_subplot(111, projection="3d")
axis.set_facecolor((1.0, 1.0, 1.0, 0.0))
mesh = Poly3DCollection(
    triangles,
    facecolors=facecolors,
    edgecolors=(0.04, 0.20, 0.36, 0.58),
    linewidths=0.55,
)
axis.add_collection3d(mesh)
axis.scatter(
    vertices[:, 0],
    vertices[:, 1],
    vertices[:, 2],
    s=7,
    color="#082f49",
    depthshade=False,
)
axis.set_xlim(-1.05, 1.05)
axis.set_ylim(-1.05, 1.05)
axis.set_zlim(-1.05, 1.05)
axis.set_box_aspect((1.0, 1.0, 1.0))
axis.view_init(elev=20.0, azim=-58.0)
axis.set_axis_off()
figure.subplots_adjust(left=0.08, right=0.92, bottom=0.0, top=1.0)
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(FIGURE_PATH, dpi=200, transparent=True)
plt.show()
plt.close(figure)
print(f"spherical README hero: {FIGURE_PATH}")


## Refresh policy

The source notebook remains output-free. Use the dedicated manual recipe to refresh the tracked image; routine notebook and CI checks do not execute this slow artifact.